# Matrix factorization experiments

These are the smaller experiments from `matrix_factorization_implementation.ipynb`, `nmf_optimization.ipynb` and `test_nmf.ipynb`. I kept them separate from the final VAE model. The first part treats zeros as missing ratings; the clustering part uses every matrix entry.

The old `SVD` and `RSVD` class names are kept below, but these two classes fit low-rank factors by gradient descent. NumPy's SVD is a different calculation.

In [ ]:
%run ./functions.ipynb
from scipy.optimize import nnls
from sklearn.datasets import make_blobs
from sklearn.preprocessing import MinMaxScaler

seed = 42
seed_everything(seed)
rng = np.random.default_rng(seed)
quick = True
iterations = 100 if quick else 1000

## SVD and masked factorization

This small binomial matrix follows my original example. For the gradient methods, a zero means an unobserved entry. This convention is only used in this section.

In [ ]:
data = rng.binomial(5, 1 / 5, (30, 20)).astype(float)
k = 5
u, sigma, vt = np.linalg.svd(data, full_matrices=False)
full_reconstruction = (u * sigma) @ vt
rank_k = (u[:, :k] * sigma[:k]) @ vt[:k]
print("Full SVD error:", np.sum((data - full_reconstruction) ** 2))
print("Rank-k error:", np.sum((data - rank_k) ** 2))

In [ ]:
class SVD:
    def __init__(self, data, k=5, alpha=0.002, max_iter=100, seed=42):
        self.data = np.asarray(data, dtype=float)
        self.mask = self.data > 0
        self.k = k
        self.alpha = alpha
        self.max_iter = max_iter
        rng = np.random.default_rng(seed)
        self.P = rng.uniform(0, 0.2, (data.shape[0], k))
        self.Q = rng.uniform(0, 0.2, (data.shape[1], k))
        self.loss_list = []

    def compute(self):
        return self.P @ self.Q.T

    def train(self):
        for step in range(self.max_iter):
            error = (self.data - self.compute()) * self.mask
            self.P += self.alpha * error @ self.Q
            error = (self.data - self.compute()) * self.mask
            self.Q += self.alpha * error.T @ self.P
            self.loss_list.append(np.sum(((self.data - self.compute()) * self.mask) ** 2))
        return self


class RSVD(SVD):
    def __init__(self, data, k=5, alpha=0.002, lmbda=0.02, max_iter=100, seed=42):
        super().__init__(data, k, alpha, max_iter, seed)
        self.mu = self.data[self.mask].mean()
        self.bu = np.zeros((data.shape[0], 1))
        self.bi = np.zeros((data.shape[1], 1))
        self.lmbda = lmbda

    def compute(self):
        return self.mu + self.bu + self.bi.T + self.P @ self.Q.T

    def train(self):
        for step in range(self.max_iter):
            error = (self.data - self.compute()) * self.mask
            self.P += self.alpha * error @ self.Q
            error = (self.data - self.compute()) * self.mask
            self.Q += self.alpha * error.T @ self.P
            error = (self.data - self.compute()) * self.mask
            # The original experiment regularized the biases only.
            self.bu += self.alpha * (error.mean(axis=1, keepdims=True) - self.lmbda * self.bu)
            self.bi += self.alpha * (error.mean(axis=0)[:, None] - self.lmbda * self.bi)
            self.loss_list.append(np.sum(((self.data - self.compute()) * self.mask) ** 2))
        return self

    def predict(self, user, item):
        return self.compute()[user, item]


modelSVD = SVD(data, k=k, max_iter=iterations, seed=seed).train()
modelRSVD = RSVD(data, k=k, max_iter=iterations, seed=seed).train()
plt.plot(modelSVD.loss_list, label="Masked MF")
plt.plot(modelRSVD.loss_list, label="With biases")
plt.xlabel("Iteration")
plt.ylabel("Squared error on observed entries")
plt.legend()
plt.show()

The loop version is useful for checking the individual updates. Its regularization is applied once per observed rating, as in the original notebook.

In [ ]:
def Matrix_decomposition(R, P, Q, alpha=0.002, beta=0.02, max_iter=100):
    P, Q = P.copy(), Q.copy()
    rows, cols = np.nonzero(R)
    loss_list = []
    for step in range(max_iter):
        for i, j in zip(rows, cols):
            error = R[i, j] - P[i] @ Q[j]
            for k in range(P.shape[1]):
                P[i, k] += alpha * (2 * error * Q[j, k] - beta * P[i, k])
                Q[j, k] += alpha * (2 * error * P[i, k] - beta * Q[j, k])
        error = R[rows, cols] - np.sum(P[rows] * Q[cols], axis=1)
        penalty = beta / 2 * (np.sum(P[rows] ** 2) + np.sum(Q[cols] ** 2))
        loss_list.append(np.sum(error ** 2) + penalty)
    return P, Q, loss_list


P = rng.uniform(0, 0.2, (data.shape[0], k))
Q = rng.uniform(0, 0.2, (data.shape[1], k))
P, Q, loss_list = Matrix_decomposition(data, P, Q, max_iter=iterations)
plt.plot(loss_list)
plt.xlabel("Iteration")
plt.ylabel("Loop SGD objective")
plt.show()

## NMF, alternating least squares and semi-NMF

Here zeros are real values. NMF constrains both factors to be nonnegative; semi-NMF only constrains the assignments `Y`. The unconstrained ALS example is included to compare the update rules.

In [ ]:
X, y = make_blobs(n_samples=240, centers=3, n_features=6, cluster_std=0.7, random_state=seed)
X_nonnegative = MinMaxScaler().fit_transform(X)
n_clusters = 3
mf_iterations = 100 if quick else 1000


def fit_multiplicative_nmf(X, k, max_iter=100, binary=None, kappa=1e-4, seed=42):
    if np.min(X) < 0:
        raise ValueError("NMF needs nonnegative input.")
    rng = np.random.default_rng(seed)
    Y = rng.uniform(0.1, 1, (len(X), k))
    C = rng.uniform(0.1, 1, (k, X.shape[1]))
    loss_list = []
    for epoch in range(max_iter):
        C *= (Y.T @ X) / (Y.T @ Y @ C + 1e-10)
        Y *= (X @ C.T) / (Y @ C @ C.T + 1e-10)
        if binary is not None:
            values = torch.as_tensor(Y, dtype=torch.float32)
            prox = improved_primp if binary == "improved" else prox_primp
            Y = prox(values, kappa).detach().cpu().numpy()
        loss_list.append(np.mean((X - Y @ C) ** 2))
    return Y, C, loss_list


def fit_als(X, k, max_iter=100, nonnegative=False, seed=42):
    rng = np.random.default_rng(seed)
    Y = rng.uniform(0.1, 1, (len(X), k))
    C = rng.uniform(0.1, 1, (k, X.shape[1]))
    loss_list = []
    for epoch in range(max_iter):
        if nonnegative:
            C = np.column_stack([nnls(Y, X[:, j])[0] for j in range(X.shape[1])])
            Y = np.vstack([nnls(C.T, row)[0] for row in X])
        else:
            C = np.linalg.lstsq(Y, X, rcond=None)[0]
            Y = np.linalg.lstsq(C.T, X.T, rcond=None)[0].T
        loss_list.append(np.mean((X - Y @ C) ** 2))
    return Y, C, loss_list


def fit_semi_nmf(X, k, max_iter=100, seed=42):
    rng = np.random.default_rng(seed)
    Y = rng.uniform(0.1, 1, (len(X), k))
    loss_list = []
    for epoch in range(max_iter):
        C = np.linalg.lstsq(Y, X, rcond=None)[0]
        XC = X @ C.T
        CC = C @ C.T
        positive_XC, negative_XC = np.maximum(XC, 0), np.maximum(-XC, 0)
        positive_CC, negative_CC = np.maximum(CC, 0), np.maximum(-CC, 0)
        Y *= np.sqrt((positive_XC + Y @ negative_CC + 1e-10) /
                     (negative_XC + Y @ positive_CC + 1e-10))
        loss_list.append(np.mean((X - Y @ C) ** 2))
    return Y, C, loss_list


factor_results = {}
factor_results["Multiplicative NMF"] = fit_multiplicative_nmf(X_nonnegative, n_clusters, mf_iterations)
factor_results["NMF + PRIMP"] = fit_multiplicative_nmf(X_nonnegative, n_clusters, mf_iterations, binary="primp")
factor_results["NMF + improved PRIMP"] = fit_multiplicative_nmf(X_nonnegative, n_clusters, mf_iterations, binary="improved")
factor_results["Nonnegative ALS"] = fit_als(X_nonnegative, n_clusters, 20 if quick else 100, nonnegative=True)
factor_results["Unconstrained ALS"] = fit_als(X_nonnegative, n_clusters, 20 if quick else 100)
factor_results["Semi-NMF"] = fit_semi_nmf(X_nonnegative, n_clusters, mf_iterations)

rows = []
for name, (Y, C, history) in factor_results.items():
    row = {"Method": name, "MSE": history[-1]}
    # Free factors have no assignment interpretation.
    if name != "Unconstrained ALS":
        row.update(score_dict(y, Y.argmax(axis=1)))
    rows.append(row)
    plt.plot(history, label=name)
display(pd.DataFrame(rows))
plt.yscale("log")
plt.xlabel("Iteration")
plt.ylabel("Reconstruction MSE")
plt.legend()
plt.show()

# Semi-NMF also accepts signed input.
Y_signed, C_signed, signed_history = fit_semi_nmf(X, n_clusters, mf_iterations)
print("Signed input, semi-NMF:", score_dict(y, Y_signed.argmax(axis=1)))

## Adam updates and the early binary penalties

The main model uses PRIMP directly. I also kept the earlier accumulated multiplier, direct adaptive-root and fixed-0.5 updates here. They are exploratory update rules, not interchangeable proximal operators for the final objective. These examples do not use labels to adjust the penalty.

The row threshold is the mean of its two largest values. `phiX` from the old notebooks was only a heuristic; the plots below use distance to binary entries and distance to a one-hot row instead.

In [ ]:
def early_binary_update(Y, lambdasX, alphaX=1e-3, prox_lr=1e-2, mode="accumulated"):
    Y = np.clip(Y, 0, 1)
    top_two = np.partition(Y, -2, axis=1)[:, -2:]
    phi = Y - top_two.mean(axis=1, keepdims=True)
    if mode == "accumulated":
        lambdasX += alphaX * phi
        Y = np.clip(Y + prox_lr * lambdasX, 0, 1)
    elif mode == "direct":
        Y = np.clip(Y + alphaX * phi, 0, 1)
    elif mode == "fixed_half":
        Y = np.clip(Y + 2 * prox_lr * lambdasX * np.where(Y > 0.5, 1, -1), 0, 1)
        phi_alt = 1 - np.abs(1 - 2 * Y)
        lambdasX += alphaX * (phi_alt - 1)
    else:
        raise ValueError(mode)
    return Y, lambdasX


def fit_early_binary_mf(X, k, epochs=100, lr=1e-2, mode="accumulated", seed=42):
    seed_everything(seed)
    data_tensor = torch.as_tensor(X, dtype=torch.float32, device=device)
    Y = nn.Parameter(torch.rand(len(X), k, device=device))
    C = nn.Parameter(torch.rand(k, X.shape[1], device=device))
    optimizerY = torch.optim.Adam([Y], lr=lr)
    optimizerC = torch.optim.Adam([C], lr=lr)
    lambdasX = np.zeros((len(X), k), dtype=np.float32)
    history = {"loss": [], "binary_error": [], "one_hot_error": []}
    for epoch in range(epochs):
        optimizerC.zero_grad()
        loss = F.mse_loss(Y.detach() @ C, data_tensor)
        loss.backward()
        optimizerC.step()
        optimizerY.zero_grad()
        loss = F.mse_loss(Y @ C.detach(), data_tensor)
        loss.backward()
        optimizerY.step()
        with torch.no_grad():
            values = Y.cpu().numpy().copy()
            values, lambdasX = early_binary_update(values, lambdasX, mode=mode)
            Y.copy_(torch.as_tensor(values, device=device))
            history["loss"].append(F.mse_loss(Y @ C, data_tensor).item())
        one_hot = np.eye(k)[values.argmax(axis=1)]
        history["binary_error"].append(np.minimum(values, 1 - values).mean())
        history["one_hot_error"].append(np.abs(values - one_hot).mean())
    return Y.detach().cpu().numpy(), C.detach().cpu().numpy(), history


adam_results = {}
for binary_mode in ["none", "primp", "improved"]:
    result = train_mf_binary_model(
        torch.tensor(X_nonnegative, dtype=torch.float32), n_clusters=n_clusters,
        epochs=mf_iterations, lr=1e-2, kappa=1e-4, gamma=0,
        binary_mode=binary_mode, seed=seed)
    adam_results[binary_mode] = result
    print(binary_mode, score_dict(y, result["pred"]))

run_early_variants = True
if run_early_variants:
    fig, axes = plt.subplots(1, 3, figsize=(13, 3))
    for mode in ["accumulated", "direct", "fixed_half"]:
        Y_early, C_early, history = fit_early_binary_mf(X_nonnegative, n_clusters, mf_iterations, mode=mode)
        print(mode, score_dict(y, Y_early.argmax(axis=1)))
        for ax, column in zip(axes, history):
            ax.plot(history[column], label=mode)
            ax.set_title(column)
            ax.set_xlabel("Iteration")
    axes[-1].legend()
    plt.tight_layout()
    plt.show()

## Symmetric NMF

For a nonnegative similarity matrix `A`, fit `A ≈ Y @ Y.T`. The original notebook formed a 6000 × 6000 Gram matrix. This example is deliberately smaller. The multiplicative step is damped to reduce the oscillation of the original full-ratio update.

In [ ]:
def fit_symmetric_nmf(A, k, max_iter=200, seed=42):
    if not np.allclose(A, A.T) or np.min(A) < 0:
        raise ValueError("Use a symmetric, nonnegative similarity matrix.")
    rng = np.random.default_rng(seed)
    Y = rng.uniform(0.1, 1, (len(A), k))
    Y *= np.sqrt(max(A.mean(), 1e-10) / max((Y @ Y.T).mean(), 1e-10))
    loss_list = []
    for epoch in range(max_iter):
        ratio = (A @ Y + 1e-10) / (Y @ (Y.T @ Y) + 1e-10)
        Y *= ratio ** 0.25
        loss_list.append(np.mean((A - Y @ Y.T) ** 2))
    return Y, loss_list


A = X_nonnegative @ X_nonnegative.T
Y_symmetric, symmetric_loss = fit_symmetric_nmf(A, n_clusters, mf_iterations)
print("Symmetric NMF:", score_dict(y, Y_symmetric.argmax(axis=1)))
plt.plot(symmetric_loss)
plt.xlabel("Iteration")
plt.ylabel("Gram reconstruction MSE")
plt.show()

## The same updates on VAE features

This optional section replaces the toy matrix with fixed VAE means, as in `nmf_optimization.ipynb` and `test_nmf.ipynb`. It trains its own VAE instead of requiring an old checkpoint. Scaling is fitted once because this is a two-stage experiment.

In [ ]:
run_mnist = False
if run_mnist:
    X_mnist, y_mnist = load_data("mnist", n_samples=2000 if quick else None, seed=seed)
    autoencoder = VariationalAutoEncoder([X_mnist.shape[1], 500, 500, 2000, 10]).to(device)
    pre_train(X_mnist, autoencoder, epochs=5 if quick else 100, seed=seed)
    Z = vae_encode(X_mnist, autoencoder)
    Z_nonnegative = MinMaxScaler().fit_transform(Z)
    Y, C, history = fit_multiplicative_nmf(Z_nonnegative, 10, mf_iterations)
    print("VAE → multiplicative NMF:", score_dict(y_mnist, Y.argmax(axis=1)))
    result = train_mf_binary_model(torch.tensor(Z_nonnegative, dtype=torch.float32),
                                  n_clusters=10, epochs=mf_iterations, seed=seed)
    print("VAE → binary MF:", score_dict(y_mnist, result["pred"]))
    centers, pred = kmeans_fun_gpu(torch.tensor(Z, dtype=torch.float32, device=device), K=10)
    print("VAE → torch KMeans:", score_dict(y_mnist, pred))